In [ ]:
!pip install selenium --quiet
!apt-get update -q
!apt-get install -y -q wget unzip

# Instala o Google Chrome
!wget -q "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"
!apt-get install -y -q ./google-chrome-stable_current_amd64.deb

# Pega a versão exata do Chrome instalado
import subprocess
chrome_version = subprocess.getoutput("google-chrome --version").split()[2]
major = chrome_version.split(".")[0]
print(f"Chrome version: {chrome_version} | Major: {major}")

# Baixa o chromedriver compatível via Chrome for Testing
import urllib.request, json
url = "https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json"
with urllib.request.urlopen(url) as r:
    data = json.load(r)

chromedriver_url = None
for v in reversed(data["versions"]):
    if v["version"].startswith(major + "."):
        for dl in v["downloads"].get("chromedriver", []):
            if dl["platform"] == "linux64":
                chromedriver_url = dl["url"]
                print(f"Chromedriver encontrado: {v['version']}")
                break
    if chromedriver_url:
        break

# Baixa e instala
!wget -q "{chromedriver_url}" -O /tmp/chromedriver.zip
!unzip -o -q /tmp/chromedriver.zip -d /tmp/
!find /tmp -name "chromedriver" -exec mv {} /usr/local/bin/chromedriver \;
!chmod +x /usr/local/bin/chromedriver
!chromedriver --version

In [ ]:
MAPA_TIMES = {
    'CAM': 'Atlético Mineiro',  'PAL': 'Palmeiras',
    'INT': 'Internacional',     'CAP': 'Athletico Paranaense',
    'CFC': 'Coritiba',          'RBB': 'Bragantino',
    'VIT': 'Vitória',           'REM': 'Remo',
    'FLU': 'Fluminense',        'GRE': 'Grêmio',
    'COR': 'Corinthians',       'BAH': 'Bahia',
    'CHA': 'Chapecoense',       'SAN': 'Santos',
    'SAO': 'São Paulo',         'FLA': 'Flamengo',
    'MIR': 'Mirassol',          'VAS': 'Vasco',
    'BOT': 'Botafogo',          'CRU': 'Cruzeiro',
    'Atlético-MG': 'Atlético Mineiro',
    'Athletico-PR': 'Athletico Paranaense',
    'Red Bull Bragantino': 'Bragantino',
    'RB Bragantino': 'Bragantino',
    'Vasco da Gama': 'Vasco',
}

In [ ]:
import time
import re
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

TEAMS = {
    "Athletico Paranaense": "https://fminside.net/clubs/7-fm-26/107206-athletico-paranaense",
    "Atlético Mineiro":     "https://fminside.net/clubs/7-fm-26/314-atletico-mineiro",
    "Bahia":                "https://fminside.net/clubs/7-fm-26/315-bahia",
    "Botafogo":             "https://fminside.net/clubs/7-fm-26/316-botafogo",
    "Bragantino":           "https://fminside.net/clubs/7-fm-26/317-bragantino",
    "Chapecoense":          "https://fminside.net/clubs/7-fm-26/301304-chapecoense",
    "Coritiba":             "https://fminside.net/clubs/7-fm-26/104776-coritiba",
    "Corinthians":          "https://fminside.net/clubs/7-fm-26/319-corinthians",
    "Cruzeiro":             "https://fminside.net/clubs/7-fm-26/321-cruzeiro",
    "Flamengo":             "https://fminside.net/clubs/7-fm-26/322-flamengo",
    "Fluminense":           "https://fminside.net/clubs/7-fm-26/323-fluminense",
    "Grêmio":               "https://fminside.net/clubs/7-fm-26/324-gremio",
    "Internacional":        "https://fminside.net/clubs/7-fm-26/326-internacional",
    "Mirassol":             "https://fminside.net/clubs/7-fm-26/301344-mirassol",
    "Palmeiras":            "https://fminside.net/clubs/7-fm-26/329-palmeiras",
    "Remo":                 "https://fminside.net/clubs/7-fm-26/334-remo",
    "Santos":               "https://fminside.net/clubs/7-fm-26/335-santos",
    "São Paulo":            "https://fminside.net/clubs/7-fm-26/337-sao-paulo",
    "Vasco":                "https://fminside.net/clubs/7-fm-26/339-vasco-da-gama",
    "Vitória":              "https://fminside.net/clubs/7-fm-26/340-vitoria",
}

print(f"{len(TEAMS)} times carregados.")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.binary_location = "/usr/bin/google-chrome"

driver = webdriver.Chrome(
    service=Service("/usr/local/bin/chromedriver"),
    options=chrome_options
)
wait = WebDriverWait(driver, 15)
print("Driver inicializado.")

In [ ]:
import time
from selenium.webdriver.common.by import By

def scroll_and_wait(driver, pause=2, max_scrolls=15):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

all_players_raw = []

for team_name, url in TEAMS.items():
    print(f"\nRaspando: {team_name}")
    try:
        driver.get(url)
        time.sleep(4)
        scroll_and_wait(driver)

        blocks = driver.find_elements(By.CSS_SELECTOR, "ul.player")

        # Primeiro passa: coleta todos os blocos COM rating, indexados por link
        players_by_link = {}

        for block in blocks:
            try:
                pos_el  = block.find_elements(By.CSS_SELECTOR, "li.position span")
                nome_el = block.find_elements(By.CSS_SELECTOR, "li.player span.name a")
                rat_el  = block.find_elements(By.CSS_SELECTOR, "li.rating span")

                if not nome_el:
                    continue

                nome    = nome_el[0].get_attribute("title").strip()
                link    = nome_el[0].get_attribute("href").strip()
                posicao = pos_el[0].text.strip().upper() if pos_el else ""
                ca_text = rat_el[0].text.strip() if rat_el else ""

                if not ca_text:
                    continue  # pula blocos sem rating

                ca = int(ca_text)

                # Guarda apenas a primeira ocorrência com rating para cada link
                if link not in players_by_link:
                    players_by_link[link] = {
                        "Nome":    nome,
                        "Time":    team_name,
                        "Posição": posicao,
                        "CA":      ca,
                        "Link":    link,
                    }

            except (ValueError, IndexError):
                continue

        team_players = list(players_by_link.values())
        all_players_raw.extend(team_players)
        print(f"{len(team_players)} jogadores coletados.")

    except Exception as e:
        print(f"Erro em {team_name}: {e}")

    time.sleep(2)

print(f"\nEtapa 1 concluída — {len(all_players_raw)} jogadores.")

In [ ]:
import time
from selenium.webdriver.common.by import By
import re

print(f"Total de jogadores para buscar PA e posição: {len(all_players_raw)}")
print("Isso pode demorar alguns minutos...\n")

for i, player in enumerate(all_players_raw):
    try:
        link = player["Link"].replace("/7-fm262/", "/7-fm-26/")
        driver.get(link)
        time.sleep(2)

        html = driver.page_source

        # ── PA ─────────────────────────────────────────────────────────────
        pa = player["CA"]  # fallback
        blocks = driver.find_elements(By.CSS_SELECTOR, "ul.player")
        for block in blocks:
            rating_spans = block.find_elements(By.CSS_SELECTOR, "li.rating span")
            if len(rating_spans) >= 2:
                ca_text = rating_spans[0].text.strip()
                pa_text = rating_spans[1].text.strip()
                if ca_text and pa_text:
                    try:
                        pa = int(pa_text)
                        break
                    except ValueError:
                        pass
            # PA dinâmico (intervalo): extrai o valor máximo do data-title
            if len(rating_spans) >= 2 and not rating_spans[1].text.strip():
                data_title = rating_spans[1].get_attribute("data-title") or ""
                match = re.search(r'(\d+)', data_title)
                if match:
                    pa = int(match.group(1))
                    break

        player["PA"] = pa

        # ── Posição (só busca se estiver vazia) ────────────────────────────
        if not player["Posição"]:
            pos_els = driver.find_elements(
                By.CSS_SELECTOR,
                "span.desktop_positions span.position.natural"
            )
            if pos_els:
                player["Posição"] = pos_els[0].text.strip().upper()

        if (i + 1) % 100 == 0:
            print(f"   {i+1}/{len(all_players_raw)} processados...")

    except Exception as e:
        player["PA"] = player["CA"]

print(f"\nEtapa 2 concluída.")

In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "Nome":           p["Nome"],
    "Time":           p["Time"],
    "Posição":        p["Posição"],
    "Rating (CA)":    p["CA"],
    "Potencial (PA)": p["PA"],
} for p in all_players_raw])

df = df.sort_values(["Time", "Rating (CA)"], ascending=[True, False]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(df.groupby("Time").size().to_string())
print()
print(df.head(10))

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

OUTPUT = "ListaDeJogadoresSerieA2026.xlsx"

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Sheet1"

headers = ["Nome", "Time", "Posição", "Rating (CA)", "Potencial (PA)"]
header_fill = PatternFill("solid", fgColor="2E75B6")
header_font = Font(bold=True, color="FFFFFF", size=11)
thin        = Side(style="thin")
border      = Border(left=thin, right=thin, top=thin, bottom=thin)

for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_idx, value=h)
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border    = border

alt_fill = PatternFill("solid", fgColor="EBF3FB")

for row_idx, row_data in enumerate(df.itertuples(index=False), 2):
    for col_idx, value in enumerate(row_data, 1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.border    = border
        cell.alignment = Alignment(horizontal="center" if col_idx >= 3 else "left")
        if row_idx % 2 == 0:
            cell.fill = alt_fill

col_widths = [30, 25, 12, 14, 16]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

ws.freeze_panes = "A2"
wb.save(OUTPUT)
print(f"{OUTPUT} salvo com {len(df)} jogadores.")

files.download(OUTPUT)